In [ ]:
import json
import pandas as pd

with open("../../dataset/train_sentiment.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../"))

In [ ]:
from utility.preprocess import clean_text
text = df['text']
sentiment = df['sentiment']

# text = df["text"].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

text_train, text_test, sentiment_train, sentiment_test = train_test_split(
    text, sentiment, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import StackingClassifier

base_models = [
    ("svm", LinearSVC()),
    ("nb", MultinomialNB()),
    # ("lr", LogisticRegression(max_iter=1000)),
    ("sgd", SGDClassifier(loss="log_loss"))
]

meta_model = LogisticRegression()

In [ ]:
stack_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5
)

### NB-SVM

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from utility.preprocess import tokenizer

vectorizer = CountVectorizer(
    tokenizer=tokenizer,
    ngram_range=(1,2),
    min_df=3
)

X_train = vectorizer.fit_transform(text_train)
X_test = vectorizer.transform(text_test)

In [ ]:
from utility.cal_nb import nb_features

r = nb_features(X_train, sentiment_train)

X_train_nb = X_train.multiply(r)
X_test_nb = X_test.multiply(r)

In [ ]:
stack_model.fit(X_train_nb, sentiment_train)

### end NB-SVM

In [ ]:
from sklearn.metrics import accuracy_score, classification_report # type: ignore

y_pred = stack_model.predict(X_test_nb)

print("Accuracy:", accuracy_score(sentiment_test, y_pred))
print(classification_report(sentiment_test, y_pred))